# LongFlow — Gate Night 4 (OUR HEAD in the healthy operating mode)

Runtime: **L4 GPU**. ~1.5 h, ~$2–3. The critical-path night: every long-form
test the 20K head ever got was under monolithic conditions now known to poison
the backbone's hidden states (register OOD, GN2/GN3). This is its first run in
the turn-split mode that cured pacing AND drift on the teacher.
Pre-registered criteria: `experiments/p1_flow_head/NOTES.md` (Gate Night 4
entry). All artifacts mirror to Drive per run; reruns skip completed work.

| cell | arm | decides |
|---|---|---|
| 4 | **A: head re-baseline** — 20K head via FlowHeadPatch, turn-split full script, euler4 AND heun8 | was E3's long-form collapse register OOD? does turn-reset rescue euler4 from the closed-loop fade? |
| 6 | **B: latency profile** — per-module timers, teacher vs our head | splits the ~122ms non-head lump; the paper's bottleneck map |
| 8 | **C: batch parity** — 4 utterances solo vs batch-4, teacher | blocking item 2; gates the 4.9× throughput claim AND the 75K cache |

Direct A/B reference: the GN3 T1 teacher renders (same script, same prompt).


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag longflow_bundle.zip + unzip"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample, heun_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/gate_night4"
DRIVE_OUT = "/content/drive/MyDrive/longflow_gate4"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

def gen_inputs_batch(texts, prompt_wavs):
    inputs = processor(text=texts, voice_samples=[[p] for p in prompt_wavs],
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_run(tag, wav, wall, extra=None):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    row = {"tag": tag, "audio_s": round(len(wav)/24000, 1), "wall_s": round(wall)}
    if extra: row.update(extra)
    report["runs"].append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night4_report.json", "w"), indent=2)
    print(row, flush=True)

def done(tag):
    if os.path.exists(f"{DRIVE_OUT}/{tag}.wav"):
        print(f"skip {tag} (already on Drive)", flush=True)
        return True
    return False

report = {"runs": []}
print("READY")


In [ ]:
# Identical script recipe to GN3 (same pool, same order) -> direct A/B vs
# the teacher's t1_turnsplit_p0 render.
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]

turns, cur, w = [], [], 0
for s in pool:
    cur.append(s); w += len(s.split())
    if w >= 60:
        turns.append("Speaker 1: " + " ".join(cur)); cur, w = [], 0
if cur:
    turns.append("Speaker 1: " + " ".join(cur))
TURNSCRIPT = "\n".join(turns) + "\n"
print(f"{sum(len(s.split()) for s in pool)} words, {len(turns)} turns")

prompts = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))
P0 = prompts[0]
report["prompt"] = os.path.basename(P0)


## 4. Arm A — the 20K head, turn-split, both samplers (the re-baseline)

Two pre-registered bets collide here:
- **Register OOD** (E3 reframe): the head collapsed on long monolithic prompts
  because those hidden states were out-of-register vs its short-clip cache.
  Turn-split keeps the backbone in the trained register → head should hold.
- **Turn-reset** (new): E1's closed-loop fade built up over ~60 s of the head
  consuming its own latents. A 60-word turn is ~25 s — each turn boundary
  re-anchors on text before the fade horizon → euler4 may survive turn-split
  even though it faded monolithically.

~50–70 min (two ~19-min renders; our head is faster per frame than teacher).


In [ ]:
ARMS = [("euler4", euler_sample, 4), ("heun8", heun_sample, 8)]
for name, sampler, nfe in ARMS:
    tag = f"a_head20_turnsplit_{name}"
    if done(tag):
        continue
    torch.manual_seed(0)
    t0 = time.time()
    with FlowHeadPatch(model, head20, mean20, std20, nfe=nfe, sway=0.0,
                       sampler=sampler) as patch, torch.inference_mode():
        out = model.generate(**gen_inputs_batch([TURNSCRIPT], [P0]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=12000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    n8 = max(len(patch.latents) // 8, 1)
    stds = [round(float(torch.cat(patch.latents[i*n8:(i+1)*n8]).std()), 3)
            for i in range(8)] if patch.latents else []
    save_run(tag, wav, time.time() - t0,
             {"head_calls": patch.calls, "head_time_s": round(patch.time_s, 1),
              "latent_std_8seg": stds, "latent_std_overall": round(float(zs.std()), 3)})


## 6. Arm B — per-module latency profile (the bottleneck map, measured)

Wraps every top-level child of `model.model` with a timer, generates ~60 s,
reports ms/frame per module — teacher head vs flow head. Splits the ~122 ms
non-head lump (LM, acoustic/semantic encoders, CFG cost shows up as LM volume).


In [ ]:
import collections
from contextlib import ExitStack

def profile_generation(patch_ctx, label, n_words=150):
    text = "Speaker 1: " + " ".join(pool)[:2000]
    script = " ".join(" ".join(pool).split()[:n_words])
    times = collections.defaultdict(float)
    hooks = []
    def wrap(name, mod):
        def pre(m, args, kwargs=None):
            m.__t0 = time.time()
        def post(m, args, out):
            times[name] += time.time() - getattr(m, "__t0", time.time())
        hooks.append(mod.register_forward_pre_hook(pre))
        hooks.append(mod.register_forward_hook(post))
    for name, mod in model.model.named_children():
        wrap(name, mod)
    torch.manual_seed(0)
    t0 = time.time()
    with ExitStack() as stack:
        p = stack.enter_context(patch_ctx()) if patch_ctx else None
        stack.enter_context(torch.inference_mode())
        out = model.generate(**gen_inputs_batch([f"Speaker 1: {script}\n"], [P0]),
                             tokenizer=processor.tokenizer, cfg_scale=1.3,
                             max_new_tokens=800)
    wall = time.time() - t0
    for h in hooks:
        h.remove()
    frames = len(out.speech_outputs[0].squeeze()) / 24000 * 7.5
    prof = {k: round(v / frames * 1000, 1) for k, v in
            sorted(times.items(), key=lambda kv: -kv[1])}
    if patch_ctx and p is not None:
        prof["flow_head_direct"] = round(p.time_s / max(p.calls, 1) * 1000, 1)
    row = {"label": label, "frames": int(frames), "wall_s": round(wall, 1),
           "ms_per_frame_total": round(wall / frames * 1000, 1), "per_module_ms": prof}
    report.setdefault("profile", []).append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night4_report.json", "w"), indent=2)
    print(json.dumps(row, indent=2), flush=True)

profile_generation(None, "teacher_head")
profile_generation(lambda: FlowHeadPatch(model, head20, mean20, std20, nfe=4,
                                         sway=0.0, sampler=euler_sample),
                   "flow_head_euler4")


## 8. Arm C — batch parity, audio-level (blocking item 2)

GN1 cell 6 showed batched capture shifts conditioning distributions; this asks
the question that matters: does batch-4 generation AUDIBLY degrade output?
Four eval texts rendered solo then as one padded batch, teacher head, seed 0.
Mac scores per-utterance WER/sim solo-vs-batched against the reseed floor.


In [ ]:
eval_files = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*.pt"))[:4]
texts, tags = [], []
for f in eval_files:
    d = torch.load(f, weights_only=True)
    texts.append(f"Speaker 1: {d['text'].strip()}\n")
    tags.append(d["utt_id"] if "utt_id" in d else os.path.basename(f)[:-3])
report["parity_texts"] = {t: x for t, x in zip(tags, texts)}

# solo renders
for tag, text in zip(tags, texts):
    rtag = f"c_solo_{tag}"
    if done(rtag):
        continue
    torch.manual_seed(0)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**gen_inputs_batch([text], [P0]),
                             tokenizer=processor.tokenizer, cfg_scale=1.3,
                             max_new_tokens=800)
    save_run(rtag, out.speech_outputs[0].detach().float().cpu().numpy().squeeze(),
             time.time() - t0)

# batch-4 render
if not done(f"c_batch_{tags[0]}"):
    torch.manual_seed(0)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**gen_inputs_batch(texts, [P0]*4),
                             tokenizer=processor.tokenizer, cfg_scale=1.3,
                             max_new_tokens=800)
    wall = time.time() - t0
    for i, tag in enumerate(tags):
        save_run(f"c_batch_{tag}",
                 out.speech_outputs[i].detach().float().cpu().numpy().squeeze(),
                 wall / 4)


## 9. Bundle

`gate_night4_bundle.zip` (zips from the Drive mirror — safe after any resume).
Mac: `unzip -o ~/Downloads/gate_night4_bundle.zip -d experiments/p1_flow_head/audio/gate_night4`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night4.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{DRIVE_OUT}/gate_night4_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night4_bundle.zip", "w") as z:
    for f in glob.glob(f"{DRIVE_OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night4_bundle.zip")
